In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/q13_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Filter customers and orders
customer_filtered = customer[["C_CUSTKEY"]]
orders_filtered = (
    orders
    [~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests")]
    [["O_ORDERKEY", "O_CUSTKEY"]]
)

# Merge and count orders per customer
df_merged = (
    customer_filtered
    .merge(orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="left")
    [["C_CUSTKEY", "O_ORDERKEY"]]
)
count_df = (
    df_merged
    .groupby("C_CUSTKEY")["O_ORDERKEY"]
    .count()
    .reset_index(name="C_COUNT")
)

# Compute distribution of customers by order count and sort
total = (
    count_df
    .groupby("C_COUNT")
    .size()
    .reset_index(name="CUSTDIST")
    .sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])
)